In [151]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import plotly.io as pio
import warnings
warnings.filterwarnings('ignore')
import yfinance as yf
import pandas as pd


from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [152]:
# ~45 SPDR industry ETFs (S&P SPDR series)
INDUSTRY_ETFS = {
    'Aerospace & Def':     'XAR',
    'Biotech':             'XBI',
    'Capital Markets':     'KCE',
    'Banks':               'KBE',
    'Regional Banks':      'KRE',
    'Insurance':           'KIE',
    'Homebuilders':        'XHB',
    'Retail':              'XRT',
    'Oil & Gas Expl':      'XOP',
    'Oil & Gas Equip':     'XES',
    'Metals & Mining':     'XME',
    'Gold Miners':         'GDX',
    'Semiconductors':      'SMH',
    'Software':            'IGV',
    'Internet':            'FDN',
    'Cloud Computing':     'SKYY',
    'Cybersecurity':       'CIBR',
    'Clean Energy':        'ICLN',
    'Solar':               'TAN',
    'Uranium':             'URA',
    'Healthcare Equip':    'XHE',
    'Pharma':              'XPH',
    'Health Providers':    'XHS',
    'Med Devices':         'IHI',
    'Transports':          'XTN',
    'Infrastructure':      'IFRA',
    'Utilities':           'XLU',
    'REITs':               'VNQ',
    'Residential REITs':   'REZ',
    'Food & Bev':          'PBJ',
    'Consumer Staples':    'XLP',
    'Leisure':             'PEJ',
    'Media':               'PBS',
    'Telecom':             'IYZ',
    'Commodities Broad':   'DJP',
    'Agriculture':         'DBA',
    'Base Metals':         'DBB',
    'US Treasuries':       'IEF',
    'HY Corporate':        'HYG',
    'IG Corporate':        'LQD',
    'TIPS':                'TIP',
    'USD Index':           'UUP',
    'EM Equity':           'EEM',
    'Europe Equity':       'VGK',
    'Japan Equity':        'EWJ',
    'China Equity':        'FXI',
}
all_tickers = []
sector_of = {}
for sector, tkrs in INDUSTRY_ETFS.items():
    all_tickers.append(tkrs)
    sector_of[tkrs] = sector

# Use it anywhere you need the name
def name(ticker):
    return sector_of.get(ticker, ticker)

In [189]:
PERIOD = '1y'
# constituents = pd.read_csv('constituents.csv')
# all_tickers = constituents['Symbol'].tolist()
print(f"Downloading {len(all_tickers)} tickers ({PERIOD})...")
data = yf.download(all_tickers, period=PERIOD, auto_adjust=True, progress=False)
closes = data['Close'].dropna(axis=1)
valid_tickers = list(closes.columns)
name_labels = [name(t) for t in valid_tickers]
print(f"Got {len(valid_tickers)} tickers, {len(closes)} trading days")

Got 45 tickers, 251 trading days


In [154]:
# ── Log returns & correlation matrix ─────────────────────────
returns = np.log(closes / closes.shift(1)).dropna().values
T, N = returns.shape
Q = T / N
gamma = N / T
print(f"N={N}, T={T}, Q={Q:.2f}, γ={gamma:.4f}")

N=45, T=250, Q=5.56, γ=0.1800


In [155]:
corr_mat = np.corrcoef(returns, rowvar=False)

# ── 2. EIGENDECOMPOSITION ────────────────────────────────────
eigenvalues, eigenvectors = np.linalg.eigh(corr_mat)
# Sort descending
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# ── 3. MARCHENKO-PASTUR ─────────────────────────────────────
def mp_density(lam, gamma):
    """MP probability density function."""
    lam_plus  = (1 + np.sqrt(gamma))**2
    lam_minus = (1 - np.sqrt(gamma))**2
    mask = (lam >= lam_minus) & (lam <= lam_plus)
    density = np.zeros_like(lam)
    density[mask] = np.sqrt((lam_plus - lam[mask]) * (lam[mask] - lam_minus)) \
                    / (2 * np.pi * gamma * lam[mask])
    return density

lam_plus  = (1 + np.sqrt(gamma))**2
lam_minus = (1 - np.sqrt(gamma))**2

sig_mask = eigenvalues > lam_plus
n_sig = sig_mask.sum()

for k in range(n_sig):
    if k == 0:
        if eigenvectors[:, k].mean() < 0:
            eigenvectors[:, k] *= -1
    else:
        if eigenvectors[:, k][np.argmax(np.abs(eigenvectors[:, k]))] < 0:
            eigenvectors[:, k] *= -1

# ── 4. EIGENVALUE CLIPPING (DENOISING) ──────────────────────
noise_eigenvalues = eigenvalues[~sig_mask]
noise_replacement = noise_eigenvalues.mean()

eigenvalues_clean = eigenvalues.copy()
eigenvalues_clean[~sig_mask] = noise_replacement

# Reconstruct: C_clean = V Λ' V^T
Lambda_clean = np.diag(eigenvalues_clean)
C_clean = eigenvectors @ Lambda_clean @ eigenvectors.T

# Renormalise diagonals back to 1
d = np.sqrt(np.diag(C_clean))
C_clean = C_clean / np.outer(d, d)

# ── 5. ABSORPTION RATIO ─────────────────────────────────────
total_var = eigenvalues.sum()
print(f"\nMP bounds: λ- = {lam_minus:.4f}, λ+ = {lam_plus:.4f}")
print(f"Significant eigenvalues: {n_sig}/{N}")
for i in range(n_sig):
    print(f"  λ{i+1:2d} = {eigenvalues[i]:.4f}  ({eigenvalues[i]/total_var*100:.1f}% of variance)")
print(f"Total variance explained by significant: {eigenvalues[sig_mask].sum()/total_var*100:.1f}%")

for k in [1, 3, 5, n_sig]:
    ar = eigenvalues[:k].sum() / total_var
    print(f"Absorption ratio (top {k:2d}): {ar*100:.1f}%")


MP bounds: λ- = 0.3315, λ+ = 2.0285
Significant eigenvalues: 4/45
  λ 1 = 15.7409  (35.0% of variance)
  λ 2 = 4.5042  (10.0% of variance)
  λ 3 = 3.4946  (7.8% of variance)
  λ 4 = 3.0267  (6.7% of variance)
Total variance explained by significant: 59.5%
Absorption ratio (top  1): 35.0%
Absorption ratio (top  3): 52.8%
Absorption ratio (top  5): 63.9%
Absorption ratio (top  4): 59.5%


In [190]:
mat_min = corr_mat.min()
mat_min_clean = C_clean.min()

fig = go.Figure(data=[go.Histogram(x=eigenvalues, nbinsx=20)])
fig.update_layout(
    template='plotly_dark',
    title=f'Histogram of Eigenvalues for Period={PERIOD} with {n_sig} significant Eigenvalues',
    xaxis_title='Eigenvalue', yaxis_title='Density',
)
fig.add_vline(x=lam_plus, line_dash='dash', line_color='red',
              annotation_text=f'λ+ = {lam_plus:.2f}')
fig.show()

fig = go.Figure(data=go.Heatmap(
    z=corr_mat,
    x=name_labels,
    y=name_labels,
    colorscale='RdBu_r',
    zmin=mat_min, zmax=1,
))

fig.update_layout(
    template='plotly_dark',
    title='Correlation Matrix',
    width=900, height=800,
)

fig.update_xaxes(tickfont=dict(size=8), dtick=1)
fig.update_yaxes(tickfont=dict(size=8), dtick=1)
fig.show()

from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, subplot_titles=['Original', 'RMT-Cleaned'])

fig.add_trace(go.Heatmap(z=corr_mat, x=name_labels, y=name_labels,
                          colorscale='RdBu_r', zmin=min(mat_min,mat_min_clean), zmax=1), row=1, col=1)
fig.add_trace(go.Heatmap(z=C_clean, x=name_labels, y=name_labels,
                          colorscale='RdBu_r', zmin=min(mat_min,mat_min_clean), zmax=1), row=1, col=2)

fig.update_layout(template='plotly_dark', width=1400, height=700)
fig.update_xaxes(tickfont=dict(size=8), dtick=1)
fig.update_yaxes(tickfont=dict(size=8), dtick=1)
fig.show()

In [196]:
# Use significant eigenvectors only (4 in your case)
coords = eigenvectors[:, :n_sig]

# Try a few k values, use elbow or silhouette to pick
sil_k_list = []
for k in range(3, 15):
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(coords)
    sil = silhouette_score(coords, labels)
    sil_k_list.append([sil,k])
    # print(f"  k={k}  silhouette={sil:.3f}")

# Pick best k and plot
best_k = max(sil_k_list, key=lambda x: x[0])[1] # or whichever had highest silhouette
km = KMeans(n_clusters=best_k, n_init=20, random_state=42)
labels = km.fit_predict(coords)

# 3D plot coloured by cluster
fig = go.Figure(data=[go.Scatter3d(
    x=eigenvectors[:, 0],
    y=eigenvectors[:, 1],
    z=eigenvectors[:, 2],
    mode='markers+text',
    text=name_labels,
    textposition='top center',
    textfont=dict(size=8),
    marker=dict(size=5, color=labels, colorscale='rainbow'),
    hovertext=[f'{name(t)} ({t}) (cluster {l})' for t, l in zip(valid_tickers, labels)],
    hoverinfo='text',
)])

fig.update_layout(
    template='plotly_dark',
    title=f'Industry ETFs — {best_k} Clusters in Eigenspace',
    scene=dict(aspectmode='cube',xaxis_title='EV1', yaxis_title='EV2', zaxis_title='EV3'),
    height=1000,width = 1200,
)
fig.show()

fig = go.Figure(data=[go.Scatter(
    x=eigenvectors[:, 0],
    y=eigenvectors[:, 1],
    mode='markers+text',
    text=name_labels,
    textposition='top center',
    textfont=dict(size=8),
    marker=dict(size=5, color=labels, colorscale='rainbow'),
    hovertext=[f'{name(t)} ({t}) (cluster {l})' for t, l in zip(valid_tickers, labels)],
    hoverinfo='text',
)])
fig.update_layout(
    template='plotly_dark',
    title=f'Industry ETFs — {best_k} Clusters in Eigenspace',
    xaxis_title='EV1', yaxis_title='EV2',
    height=1000,width = 1200,
)
fig.show()

# Print cluster members
for c in range(best_k):
    members = [name(t) for t, l in zip(valid_tickers, labels) if l == c]
    print(f"\nCluster {c}: {members}")


Cluster 0: ['Cybersecurity', 'Internet', 'Software', 'Cloud Computing']

Cluster 1: ['HY Corporate', 'Med Devices', 'Banks', 'Capital Markets', 'Insurance', 'Regional Banks', 'Leisure', 'Healthcare Equip', 'Health Providers', 'Pharma', 'Retail', 'Transports']

Cluster 2: ['Base Metals', 'EM Equity', 'Japan Equity', 'China Equity', 'Gold Miners', 'Clean Energy', 'Telecom', 'Semiconductors', 'Solar', 'Uranium', 'Europe Equity', 'Aerospace & Def', 'Biotech', 'Metals & Mining']

Cluster 3: ['Infrastructure', 'Food & Bev', 'Residential REITs', 'REITs', 'Homebuilders', 'Consumer Staples', 'Utilities']

Cluster 4: ['US Treasuries', 'IG Corporate', 'TIPS']

Cluster 5: ['Agriculture', 'Commodities Broad', 'USD Index', 'Oil & Gas Equip', 'Oil & Gas Expl']


In [157]:
from scipy.spatial.distance import pdist, squareform
coords = eigenvectors[:, :n_sig]

dist = pd.DataFrame(squareform(pdist(coords)), index=valid_tickers, columns=valid_tickers)

pairs = []
for i in range(len(valid_tickers)):
    for j in range(i+1, len(valid_tickers)):
        pairs.append((valid_tickers[i], valid_tickers[j], dist.iloc[i, j]))
pairs.sort(key=lambda x: x[2])
print(f"Closest pairs in {n_sig}D eigenspace:")
for t1, t2, d2 in pairs[:15]:
    print(f"  {name(t1):20s}({t1}) ↔ {name(t2):20s}({t2})  D={d2:.4f}")

print(f"Furthest pairs in {n_sig}D eigenspace:")
for t1, t2, d2 in pairs[-15:]:
    print(f"  {name(t1):20s}({t1}) ↔ {name(t2):20s}({t2})  D={d2:.4f} ")

Closest pairs in 4D eigenspace:
  Banks               (KBE) ↔ Regional Banks      (KRE)  D=0.0192
  Retail              (XRT) ↔ Transports          (XTN)  D=0.0318
  Software            (IGV) ↔ Cloud Computing     (SKYY)  D=0.0425
  Internet            (FDN) ↔ Cloud Computing     (SKYY)  D=0.0436
  Cybersecurity       (CIBR) ↔ Software            (IGV)  D=0.0517
  Clean Energy        (ICLN) ↔ Metals & Mining     (XME)  D=0.0535
  Cybersecurity       (CIBR) ↔ Cloud Computing     (SKYY)  D=0.0570
  Cybersecurity       (CIBR) ↔ Internet            (FDN)  D=0.0595
  Leisure             (PEJ) ↔ Healthcare Equip    (XHE)  D=0.0615
  Healthcare Equip    (XHE) ↔ Health Providers    (XHS)  D=0.0620
  REITs               (VNQ) ↔ Homebuilders        (XHB)  D=0.0636
  Base Metals         (DBB) ↔ China Equity        (FXI)  D=0.0663
  Clean Energy        (ICLN) ↔ Uranium             (URA)  D=0.0710
  Internet            (FDN) ↔ Software            (IGV)  D=0.0712
  Med Devices         (IHI) ↔ Health